<a href="https://colab.research.google.com/github/Pohyi118/umhackathon/blob/main/datapreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install pdfplumber pandas
!pip install yfinance

the setup and ingestion

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mindweavetech/australian-sme-business-dataset")

print("Path to dataset files:", path)

100%|██████████| 142k/142k [00:00<00:00, 62.5MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/mindweavetech/australian-sme-business-dataset/versions/3


In [12]:
import os
os.listdir('/root/.cache/kagglehub/datasets/mindweavetech/australian-sme-business-dataset/versions/3')


['bank_accounts.csv',
 'tax_returns.csv',
 'chart_of_accounts.csv',
 'companies.csv',
 'sales_invoices_sample.csv',
 'pay_run_lines_sample.csv',
 'pay_runs.csv',
 'suppliers.csv',
 'sales_order_lines_sample.csv',
 'README.md',
 'price_history.csv',
 'journal_entries_sample.csv',
 'sales_orders_sample.csv',
 'employees.csv',
 'purchase_orders_sample.csv',
 'customers.csv',
 'fixed_assets.csv',
 'bank_statements.csv',
 'journal_entry_lines_sample.csv',
 '_relationships.csv',
 'inventory_movements_sample.csv',
 'tax_return_lines.csv',
 'products.csv',
 '_row_counts.csv',
 'projects.csv',
 'departments.csv',
 'bank_transactions_sample.csv']

In [14]:
import os

folder_path = '/root/.cache/kagglehub/datasets/mindweavetech/australian-sme-business-dataset/versions/3/'
print("The files inside this folder are:")
print(os.listdir(folder_path))

The files inside this folder are:
['bank_accounts.csv', 'tax_returns.csv', 'chart_of_accounts.csv', 'companies.csv', 'sales_invoices_sample.csv', 'pay_run_lines_sample.csv', 'pay_runs.csv', 'suppliers.csv', 'sales_order_lines_sample.csv', 'README.md', 'price_history.csv', 'journal_entries_sample.csv', 'sales_orders_sample.csv', 'employees.csv', 'purchase_orders_sample.csv', 'customers.csv', 'fixed_assets.csv', 'bank_statements.csv', 'journal_entry_lines_sample.csv', '_relationships.csv', 'inventory_movements_sample.csv', 'tax_return_lines.csv', 'products.csv', '_row_counts.csv', 'projects.csv', 'departments.csv', 'bank_transactions_sample.csv']


layer 1:the hr skeleton

In [15]:
import pandas as pd

# Define the base folder path
folder_path = '/root/.cache/kagglehub/datasets/mindweavetech/australian-sme-business-dataset/versions/3/'

# 1. Load the HR Skeleton (Who is doing the work)
df_employees = pd.read_csv(folder_path + 'employees.csv')

# 2. Load the Sales Engine (Where the revenue comes from)
df_sales = pd.read_csv(folder_path + 'sales_orders_sample.csv')

# 3. Load the Logistics Load (Where the bottlenecks happen)
df_inventory = pd.read_csv(folder_path + 'inventory_movements_sample.csv')

print(f"Loaded {len(df_employees)} Employees")
print(f"Loaded {len(df_sales)} Sales Orders")
print(f"Loaded {len(df_inventory)} Inventory Movements")

# Let's peek at the sales data to see what we can localize
display(df_sales.head(2))

Loaded 14 Employees
Loaded 200 Sales Orders
Loaded 200 Inventory Movements


,id,company_id,customer_id,order_number,order_date,status,subtotal,gst_amount,total_amount,notes
0,9c0e1c03-8303-4011-92e9-1ef93b327150,c3e61d8e-e415-4326-931a-cffe4ec7ba0f,6fd6968c-c8e8-4aab-a786-3f1142204c45,SO-000001,2023-07-03,confirmed,1718.70,171.87,1890.57,NaN
1,59eacb39-fcd7-463e-927d-a620c96c948f,c3e61d8e-e415-4326-931a-cffe4ec7ba0f,990f257f-efe9-4741-b0a0-84bcf874ac1d,SO-000002,2023-07-03,confirmed,882.13,88.22,970.35,NaN


In [18]:
import pandas as pd

# Define the base folder path
folder_path = '/root/.cache/kagglehub/datasets/mindweavetech/australian-sme-business-dataset/versions/3/'

# 1. Load the HR Skeleton (Who is doing the work)
df_employees = pd.read_csv(folder_path + 'employees.csv')

# 2. Load the Sales Engine (Where the revenue comes from)
df_sales = pd.read_csv(folder_path + 'sales_orders_sample.csv')

# 3. Load the Logistics Load (Where the bottlenecks happen)
df_inventory = pd.read_csv(folder_path + 'inventory_movements_sample.csv')

print(f"Loaded {len(df_employees)} Employees")
print(f"Loaded {len(df_sales)} Sales Orders")
print(f"Loaded {len(df_inventory)} Inventory Movements")

# Let's peek at the sales data to see what we can localize
display(df_sales.head(2))


Loaded 14 Employees
Loaded 200 Sales Orders
Loaded 200 Inventory Movements


,id,company_id,customer_id,order_number,order_date,status,subtotal,gst_amount,total_amount,notes
0,9c0e1c03-8303-4011-92e9-1ef93b327150,c3e61d8e-e415-4326-931a-cffe4ec7ba0f,6fd6968c-c8e8-4aab-a786-3f1142204c45,SO-000001,2023-07-03,confirmed,1718.70,171.87,1890.57,NaN
1,59eacb39-fcd7-463e-927d-a620c96c948f,c3e61d8e-e415-4326-931a-cffe4ec7ba0f,990f257f-efe9-4741-b0a0-84bcf874ac1d,SO-000002,2023-07-03,confirmed,882.13,88.22,970.35,NaN


In [20]:
print(df_employees.columns.tolist())

['id', 'company_id', 'department_id', 'employee_number', 'first_name', 'last_name', 'email', 'phone', 'date_of_birth', 'gender', 'tax_file_number', 'start_date', 'termination_date', 'termination_reason', 'status', 'employment_type', 'role', 'annual_salary', 'hourly_rate', 'super_fund', 'super_member_number', 'street', 'city', 'state', 'postcode', 'bank_bsb', 'bank_account_number', 'bank_account_name']


localising the sales engine

In [21]:
import pandas as pd
import numpy as np

# 1. The 99 Speedmart Localization Function
# We look at the 'role' column and assign a realistic RM monthly salary
def localize_salary_to_rm(role_title):
    role = str(role_title).lower()

    if 'manager' in role or 'director' in role or 'head' in role:
        # Management tier
        return np.random.randint(5000, 12000)

    elif 'executive' in role or 'admin' in role or 'accountant' in role or 'sales' in role:
        # Mid-level office and sales staff
        return np.random.randint(3000, 5000)

    else:
        # General Staff / Warehouse (Grounded in the RM1,700 minimum wage and 99 Speedmart average)
        return np.random.randint(1700, 2800)

# 2. Apply the localization to the dataset
df_employees['monthly_salary_rm'] = df_employees['role'].apply(localize_salary_to_rm)

# 3. Calculate EPF (Employer Contribution - 13% for under 5k, 12% for over 5k)
df_employees['projected_epf_rm'] = df_employees['monthly_salary_rm'].apply(
    lambda x: round(x * 0.13, 2) if x <= 5000 else round(x * 0.12, 2)
)

# 4. Strip away the useless Australian data (like super_fund, bank_bsb, etc.)
# We only keep the columns the Digital Twin needs to function
columns_to_keep = [
    'id',
    'department_id',
    'first_name',
    'role',
    'employment_type',
    'monthly_salary_rm',
    'projected_epf_rm'
]

df_hr_clean = df_employees[columns_to_keep]

# 5. Show the new localized HR skeleton!
print("Successfully created the Malaysian HR Skeleton.")
display(df_hr_clean.head())

Successfully created the Malaysian HR Skeleton.


,id,department_id,first_name,role,employment_type,monthly_salary_rm,projected_epf_rm
0,9b2d8a13-52d0-46ef-af6d-640ac2453d12,b4fd80b3-d2f0-4cef-ba97-d5a93a6ec5b5,Cheyenne,Assistant Manager,full_time,9188,1102.56
1,5197f5d3-738d-4e77-8916-31de3d47080a,9f7a921e-dc39-4c56-bcf0-d96d44d8e310,Suzanne,Warehouse Associate,part_time,2786,362.18
2,9862a9eb-96ca-46f8-a3b9-a6d06d65e53d,b4fd80b3-d2f0-4cef-ba97-d5a93a6ec5b5,Devin,Assistant Manager,part_time,8407,1008.84
3,016df4b3-8506-46fc-ad24-b734ce32aaaa,b4fd80b3-d2f0-4cef-ba97-d5a93a6ec5b5,Julie,Store Manager,full_time,6435,772.20
4,438b2ff9-dc5e-4891-b962-59ee8d479711,1090847c-48f2-4359-88b9-c62a9824b956,Erika,Accounts Officer,full_time,2381,309.53


connecting hr to operations
his will find the employees who work on the floor (Managers, Associates, Sales) and randomly assign the localized sales orders to them.

In [23]:
import pandas as pd
import numpy as np

# --- STEP 1: CREATE df_sales_clean (Localize the Sales) ---

# Rename 'gst_amount' to 'sst_amount' to reflect Malaysian tax
df_sales_localized = df_sales.rename(columns={'gst_amount': 'sst_amount'})

# Apply RM conversion multiplier to make the volume look like a Malaysian SME
conversion_multiplier = 3.2
df_sales_localized['subtotal_rm'] = (df_sales_localized['subtotal'] * conversion_multiplier).round(2)
df_sales_localized['sst_amount_rm'] = (df_sales_localized['sst_amount'] * conversion_multiplier).round(2)
df_sales_localized['total_amount_rm'] = (df_sales_localized['total_amount'] * conversion_multiplier).round(2)

# --- STEP 2: LINK TO HR SKELETON ---

# Identify the Frontline/Sales Staff in your new HR table
target_roles = ['manager', 'sales', 'associate', 'cashier']
sales_staff = df_hr_clean[df_hr_clean['role'].str.lower().str.contains('|'.join(target_roles))]
sales_staff_ids = sales_staff['id'].tolist()

# Assign the Sales Orders to these Employees
if sales_staff_ids:
    # Randomly distribute the 200 sales orders among your frontline staff
    df_sales_localized['handled_by_employee_id'] = np.random.choice(sales_staff_ids, size=len(df_sales_localized))
    print(f"Successfully linked {len(df_sales_localized)} sales orders to {len(sales_staff_ids)} frontline staff members.")
else:
    print("Warning: Could not find any frontline staff to assign orders to.")

# --- STEP 3: CLEAN UP THE TABLE ---
# Keep only the localized RM columns and the new employee link
columns_to_keep = ['order_number', 'order_date', 'status', 'subtotal_rm', 'sst_amount_rm', 'total_amount_rm', 'handled_by_employee_id']
df_sales_clean = df_sales_localized[columns_to_keep]

# Display the final connected table
display(df_sales_clean.head(5))

Successfully linked 200 sales orders to 9 frontline staff members.


,order_number,order_date,status,subtotal_rm,sst_amount_rm,total_amount_rm,handled_by_employee_id
0,SO-000001,2023-07-03,confirmed,5499.84,549.98,6049.82,d6c24227-7c06-4363-8496-04f9649e3bcd
1,SO-000002,2023-07-03,confirmed,2822.82,282.30,3105.12,4d6d5ebe-289a-4427-a5ff-5459119db731
2,SO-000003,2023-07-04,confirmed,11036.83,1103.68,12140.51,016df4b3-8506-46fc-ad24-b734ce32aaaa
3,SO-000004,2023-07-04,confirmed,1473.47,147.36,1620.83,4d6d5ebe-289a-4427-a5ff-5459119db731
4,SO-000005,2023-07-05,confirmed,7849.12,784.93,8634.05,016df4b3-8506-46fc-ad24-b734ce32aaaa


In [24]:
import pandas as pd
import numpy as np

# 1. Identify the Frontline/Sales Staff in your new HR table
# We look for roles that would handle orders or store operations
target_roles = ['manager', 'sales', 'associate', 'cashier']
sales_staff = df_hr_clean[df_hr_clean['role'].str.lower().str.contains('|'.join(target_roles))]

sales_staff_ids = sales_staff['id'].tolist()

# 2. Assign the Sales Orders to these Employees
# Assuming you still have df_sales_clean from the previous step in memory
if sales_staff_ids:
    # Randomly distribute the 200 sales orders among your frontline staff
    df_sales_clean['handled_by_employee_id'] = np.random.choice(sales_staff_ids, size=len(df_sales_clean))
    print(f"Successfully linked {len(df_sales_clean)} sales orders to {len(sales_staff_ids)} frontline staff members.")
else:
    print("Warning: Could not find any frontline staff to assign orders to.")

# 3. Preview the final connected Sales Engine
display(df_sales_clean.head(5))

Successfully linked 200 sales orders to 9 frontline staff members.


/tmp/ipykernel_2086/703313864.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sales_clean['handled_by_employee_id'] = np.random.choice(sales_staff_ids, size=len(df_sales_clean))


,order_number,order_date,status,subtotal_rm,sst_amount_rm,total_amount_rm,handled_by_employee_id
0,SO-000001,2023-07-03,confirmed,5499.84,549.98,6049.82,5197f5d3-738d-4e77-8916-31de3d47080a
1,SO-000002,2023-07-03,confirmed,2822.82,282.30,3105.12,d6c24227-7c06-4363-8496-04f9649e3bcd
2,SO-000003,2023-07-04,confirmed,11036.83,1103.68,12140.51,9b2d8a13-52d0-46ef-af6d-640ac2453d12
3,SO-000004,2023-07-04,confirmed,1473.47,147.36,1620.83,9b2d8a13-52d0-46ef-af6d-640ac2453d12
4,SO-000005,2023-07-05,confirmed,7849.12,784.93,8634.05,9862a9eb-96ca-46f8-a3b9-a6d06d65e53d


We need to load that inventory_movements_sample.csv file you downloaded earlier, and link those physical movements to your Warehouse staff. This will allow your system to calculate exactly how many boxes a single warehouse worker has to pack per day based on the sales volume.

In [26]:
print(df_inventory.columns.tolist())


['id', 'product_id', 'movement_date', 'movement_type', 'quantity', 'unit_cost', 'reference_type', 'reference_id', 'balance_after']


In [28]:
import pandas as pd
import numpy as np

# 1. Load the Logistics Data
folder_path = '/root/.cache/kagglehub/datasets/mindweavetech/australian-sme-business-dataset/versions/3/'
df_inventory = pd.read_csv(folder_path + 'inventory_movements_sample.csv')

# 2. Identify the Warehouse/Logistics Staff in your HR table
# (Assuming df_hr_clean is still in memory from our previous step)
target_warehouse_roles = ['warehouse', 'logistics', 'inventory', 'stock']
warehouse_staff = df_hr_clean[df_hr_clean['role'].str.lower().str.contains('|'.join(target_warehouse_roles))]
warehouse_staff_ids = warehouse_staff['id'].tolist()

# 3. Assign the Physical Workload
if warehouse_staff_ids:
    # Filter for inventory movements tied to sales orders using 'reference_type'
    df_outbound = df_inventory[df_inventory['reference_type'].str.contains('sales', case=False, na=False)].copy()

    # Randomly assign these packing/shipping tasks to your warehouse team
    df_outbound['packed_by_employee_id'] = np.random.choice(warehouse_staff_ids, size=len(df_outbound))

    print(f"Successfully assigned {len(df_outbound)} outbound shipments to {len(warehouse_staff_ids)} warehouse staff members.")
else:
    print("Warning: Could not find any Warehouse staff in df_hr_clean to assign tasks to. Check your roles!")

# 4. Clean up the Logistics table for the Digital Twin
# We just need to know the date, the quantity moved, and who moved it
columns_to_keep = ['movement_date', 'quantity', 'packed_by_employee_id']
df_logistics_clean = df_outbound[columns_to_keep]

display(df_logistics_clean.head(5))


Successfully assigned 192 outbound shipments to 2 warehouse staff members.


,movement_date,quantity,packed_by_employee_id
0,2023-07-03,-10,c06dbd8c-46d0-4a05-a123-c65c960a53e8
1,2023-07-03,-1,5197f5d3-738d-4e77-8916-31de3d47080a
2,2023-07-03,-1,c06dbd8c-46d0-4a05-a123-c65c960a53e8
3,2023-07-03,-7,c06dbd8c-46d0-4a05-a123-c65c960a53e8
4,2023-07-03,-9,c06dbd8c-46d0-4a05-a123-c65c960a53e8


In [29]:
import json
from google.colab import files

# 1. Convert all your clean DataFrames into Python dictionaries
hr_data = df_hr_clean.to_dict(orient='records')
sales_data = df_sales_clean.to_dict(orient='records')
logistics_data = df_logistics_clean.to_dict(orient='records')

# 2. Architect the Master JSON Payload
# This structure makes it incredibly easy for React/D3.js to read the data
nadi_pekerja_master_twin = {
    "metadata": {
        "simulation_type": "Retail_Distribution_Digital_Twin",
        "currency": "MYR",
        "tax_structure": "SST"
    },
    "workforce": hr_data,
    "revenue_engine": sales_data,
    "operational_load": logistics_data
}

# 3. Save the payload as a JSON file in the Colab environment
output_filename = 'nadi_pekerja_master_twin.json'
with open(output_filename, 'w') as f:
    json.dump(nadi_pekerja_master_twin, f, indent=4)

print(f"Data successfully packaged! Downloading {output_filename}...")

# 4. Trigger the download to your physical laptop
files.download(output_filename)


Data successfully packaged! Downloading nadi_pekerja_master_twin.json...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

standardization and localization(heavy lifting)